System information (for reproducibility):

In [3]:
versioninfo()

Julia Version 1.12.5
Commit 5fe89b8ddc1 (2026-02-09 16:05 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 10 × Apple M1 Pro
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m1)
  GC: Built with stock GC
Threads: 1 default, 1 interactive, 1 GC (on 8 virtual cores)


Load packages:

In [4]:
using Pkg

Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Desktop/UCLA/Biostat 257/257-hw/hw7`


Status `~/Desktop/UCLA/Biostat 257/257-hw/hw7/Project.toml`
  [6e4b80f9] BenchmarkTools v1.8.0
  [31c24e10] Distributions v0.25.125
  [37e2e46d] LinearAlgebra v1.12.0
  [9a3f8284] Random v1.11.0


Again we continue with the linear mixed effects model (LMM)
$$
    \mathbf{Y}_i = \mathbf{X}_i \boldsymbol{\beta} + \mathbf{Z}_i \boldsymbol{\gamma}_i + \boldsymbol{\epsilon}_i, \quad i=1,\ldots,n,
$$
where   
- $\mathbf{Y}_i \in \mathbb{R}^{n_i}$ is the response vector of $i$-th individual,  
- $\mathbf{X}_i \in \mathbb{R}^{n_i \times p}$ is the fixed effects predictor matrix of $i$-th individual,  
- $\mathbf{Z}_i \in \mathbb{R}^{n_i \times q}$ is the random effects predictor matrix of $i$-th individual,  
- $\boldsymbol{\epsilon}_i \in \mathbb{R}^{n_i}$ are multivariate normal $N(\mathbf{0}_{n_i},\sigma^2 \mathbf{I}_{n_i})$,  
- $\boldsymbol{\beta} \in \mathbb{R}^p$ are fixed effects, and  
- $\boldsymbol{\gamma}_i \in \mathbb{R}^q$ are random effects assumed to be $N(\mathbf{0}_q, \boldsymbol{\Sigma}_{q \times q}$) independent of $\boldsymbol{\epsilon}_i$.

The log-likelihood of the $i$-th datum $(\mathbf{y}_i, \mathbf{X}_i, \mathbf{Z}_i)$ is 
$$
    \ell_i(\boldsymbol{\beta}, \mathbf{L}, \sigma^2) = - \frac{n_i}{2} \log (2\pi) - \frac{1}{2} \log \det \boldsymbol{\Omega}_i - \frac{1}{2} (\mathbf{y} - \mathbf{X}_i \boldsymbol{\beta})^T \boldsymbol{\Omega}_i^{-1} (\mathbf{y} - \mathbf{X}_i \boldsymbol{\beta}),
$$
where
$$
    \boldsymbol{\Omega}_i = \sigma^2 \mathbf{I}_{n_i} + \mathbf{Z}_i \boldsymbol{\Sigma} \mathbf{Z}_i^T.
$$
Given $m$ independent data points $(\mathbf{y}_i, \mathbf{X}_i, \mathbf{Z}_i)$, $i=1,\ldots,m$, we seek the maximum likelihood estimate (MLE) by maximizing the log-likelihood
$$
\ell(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2) = \sum_{i=1}^m \ell_i(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2).
$$

In HW6, we used the nonlinear programming (NLP) approach (Newton type algorithms) for optimization. In this assignment, we derive and implement an expectation-maximization (EM) algorithm for the same problem.

In [5]:
# load necessary packages; make sure install them first
using BenchmarkTools, Distributions, LinearAlgebra, Random

## Q1. (10 pts) Refresher on normal-normal model

Assume the conditional distribution
$$
\mathbf{y} \mid \boldsymbol{\gamma} \sim N(\mathbf{X} \boldsymbol{\beta} + \mathbf{Z} \boldsymbol{\gamma}, \sigma^2 \mathbf{I}_n)
$$
and the prior distribution
$$
\boldsymbol{\gamma} \sim N(\mathbf{0}_q, \boldsymbol{\Sigma}).
$$
By the Bayes theorem, the posterior distribution is
\begin{eqnarray*}
f(\boldsymbol{\gamma} \mid \mathbf{y}) &=& \frac{f(\mathbf{y} \mid \boldsymbol{\gamma}) \times f(\boldsymbol{\gamma})}{f(\mathbf{y})}, \end{eqnarray*}
where $f$ denotes corresponding density. 

Show that the posterior distribution of random effects $\boldsymbol{\gamma}$ is a multivariate normal with mean
\begin{eqnarray*}
\mathbb{E} (\boldsymbol{\gamma} \mid \mathbf{y}) &=& \sigma^{-2} (\sigma^{-2} \mathbf{Z}^T \mathbf{Z} + \boldsymbol{\Sigma}^{-1})^{-1 } \mathbf{Z}^T (\mathbf{y} - \mathbf{X} \boldsymbol{\beta}) \\
&=& \boldsymbol{\Sigma} \mathbf{Z}^T (\mathbf{Z} \boldsymbol{\Sigma} \mathbf{Z}^T + \sigma^2 \mathbf{I})^{-1} (\mathbf{y} - \mathbf{X} \boldsymbol{\beta})
\end{eqnarray*}
and covariance
\begin{eqnarray*}
\text{Var} (\boldsymbol{\gamma} \mid \mathbf{y}) &=& (\sigma^{-2} \mathbf{Z}^T \mathbf{Z} + \boldsymbol{\Sigma}^{-1})^{-1} \\
&=& \boldsymbol{\Sigma} - \boldsymbol{\Sigma} \mathbf{Z}^T (\mathbf{Z} \boldsymbol{\Sigma} \mathbf{Z}^T + \sigma^2 \mathbf{I})^{-1} \mathbf{Z} \boldsymbol{\Sigma}.
\end{eqnarray*}

**Answer:** Up to constants not involving $\boldsymbol{\gamma}$, Bayes' theorem gives
$$\log f(\boldsymbol{\gamma} \mid \mathbf{y}) \propto \log f(\mathbf{y} \mid \boldsymbol{\gamma}) + \log f(\boldsymbol{\gamma}).$$
The likelihood and prior contribute
$$\log f(\mathbf{y} \mid \boldsymbol{\gamma}) = -\frac{1}{2\sigma^2}|\mathbf{y} - \mathbf{X}\boldsymbol{\beta} - \mathbf{Z}\boldsymbol{\gamma}|^2 + \text{const},$$
$$\log f(\boldsymbol{\gamma}) = -\frac{1}{2}\boldsymbol{\gamma}^T \boldsymbol{\Sigma}^{-1} \boldsymbol{\gamma} + \text{const}.$$
Let $\mathbf{r} = \mathbf{y} - \mathbf{X}\boldsymbol{\beta}$. Expanding the squared norm and dropping all terms free of $\boldsymbol{\gamma}$, the log-kernel of the posterior is
$$-\frac{1}{2}\left[\boldsymbol{\gamma}^T\left(\sigma^{-2}\mathbf{Z}^T\mathbf{Z} + \boldsymbol{\Sigma}^{-1}\right)\boldsymbol{\gamma} - 2\boldsymbol{\gamma}^T\sigma^{-2}\mathbf{Z}^T\mathbf{r}\right].$$

Define
$$\mathbf{V} = \left(\sigma^{-2}\mathbf{Z}^T\mathbf{Z} + \boldsymbol{\Sigma}^{-1}\right)^{-1}, \qquad \boldsymbol{\mu} = \sigma^{-2}\mathbf{V}\mathbf{Z}^T\mathbf{r}.$$
Using the completing-the-square identity with $\mathbf{A} = \mathbf{V}^{-1}$ and $\mathbf{b} = \sigma^{-2}\mathbf{Z}^T\mathbf{r}$,
$$\boldsymbol{\gamma}^T\mathbf{A}\boldsymbol{\gamma} - 2\boldsymbol{\gamma}^T\mathbf{b} = (\boldsymbol{\gamma} - \boldsymbol{\mu})^T\mathbf{A}(\boldsymbol{\gamma} - \boldsymbol{\mu}) - \boldsymbol{\mu}^T\mathbf{A}\boldsymbol{\mu},$$
the last term is absorbed into the constant, leaving
$$\log f(\boldsymbol{\gamma} \mid \mathbf{y}) = -\frac{1}{2}(\boldsymbol{\gamma} - \boldsymbol{\mu})^T \mathbf{V}^{-1}(\boldsymbol{\gamma} - \boldsymbol{\mu}) + \text{const}.$$
This is the log-kernel of a multivariate normal, so $\boldsymbol{\gamma} \mid \mathbf{y} \sim N(\boldsymbol{\mu}, \mathbf{V})$, with
$$\mathbb{E}(\boldsymbol{\gamma} \mid \mathbf{y}) = \sigma^{-2}\left(\sigma^{-2}\mathbf{Z}^T\mathbf{Z} + \boldsymbol{\Sigma}^{-1}\right)^{-1}\mathbf{Z}^T(\mathbf{y} - \mathbf{X}\boldsymbol{\beta}), \qquad \operatorname{Var}(\boldsymbol{\gamma} \mid \mathbf{y}) = \left(\sigma^{-2}\mathbf{Z}^T\mathbf{Z} + \boldsymbol{\Sigma}^{-1}\right)^{-1}.$$

The Woodbury matrix identity states
$$\left(\mathbf{A} + \mathbf{U}\mathbf{C}\mathbf{V}\right)^{-1} = \mathbf{A}^{-1} - \mathbf{A}^{-1}\mathbf{U}\left(\mathbf{C}^{-1} + \mathbf{V}\mathbf{A}^{-1}\mathbf{U}\right)^{-1}\mathbf{V}\mathbf{A}^{-1}.$$
Setting $\mathbf{A} = \boldsymbol{\Sigma}^{-1}$, $\mathbf{U} = \sigma^{-2}\mathbf{Z}^T$, $\mathbf{C} = \mathbf{I}$, $\mathbf{V} = \mathbf{Z}$ yields
$$\left(\sigma^{-2}\mathbf{Z}^T\mathbf{Z} + \boldsymbol{\Sigma}^{-1}\right)^{-1} = \boldsymbol{\Sigma} - \boldsymbol{\Sigma}\mathbf{Z}^T\left(\mathbf{Z}\boldsymbol{\Sigma}\mathbf{Z}^T + \sigma^2\mathbf{I}\right)^{-1}\mathbf{Z}\boldsymbol{\Sigma},$$
which is the alternative form for $\operatorname{Var}(\boldsymbol{\gamma} \mid \mathbf{y})$.
For the alternative mean, note the identity
$$\left(\sigma^{-2}\mathbf{Z}^T\mathbf{Z} + \boldsymbol{\Sigma}^{-1}\right)\boldsymbol{\Sigma}\mathbf{Z}^T = \sigma^{-2}\mathbf{Z}^T\left(\mathbf{Z}\boldsymbol{\Sigma}\mathbf{Z}^T + \sigma^2\mathbf{I}\right),$$
which can be verified by expanding both sides. Multiplying on the left by $\mathbf{V}$ and on the right by $\left(\mathbf{Z}\boldsymbol{\Sigma}\mathbf{Z}^T + \sigma^2\mathbf{I}\right)^{-1}$ gives
$$\mathbf{V}\cdot\sigma^{-2}\mathbf{Z}^T = \boldsymbol{\Sigma}\mathbf{Z}^T\left(\mathbf{Z}\boldsymbol{\Sigma}\mathbf{Z}^T + \sigma^2\mathbf{I}\right)^{-1},$$
and therefore
$$\mathbb{E}(\boldsymbol{\gamma} \mid \mathbf{y}) = \boldsymbol{\Sigma}\mathbf{Z}^T\left(\mathbf{Z}\boldsymbol{\Sigma}\mathbf{Z}^T + \sigma^2\mathbf{I}\right)^{-1}(\mathbf{y} - \mathbf{X}\boldsymbol{\beta}). \qquad \square$$


## Q2. (20 pts) Derive EM algorithm

1. Write down the complete log-likelihood
$$
\sum_{i=1}^m \log f(\mathbf{y}_i, \boldsymbol{\gamma}_i \mid \boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2)
$$

2. Derive the $Q$ function (E-step).
$$
Q(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2 \mid \boldsymbol{\beta}^{(t)}, \boldsymbol{\Sigma}^{(t)}, \sigma^{2(t)}).
$$

3. Derive the EM (or ECM) update of $\boldsymbol{\beta}$, $\boldsymbol{\Sigma}$, and $\sigma^2$ (M-step). 

**Answer:** We have $m$ independent groups. For group $i$:
$$\mathbf{y}_i \mid \boldsymbol{\gamma}_i \sim N(\mathbf{X}_i\boldsymbol{\beta} + \mathbf{Z}_i\boldsymbol{\gamma}_i,\ \sigma^2\mathbf{I}_{n_i}), \qquad \boldsymbol{\gamma}_i \sim N(\mathbf{0},\boldsymbol{\Sigma}),$$
independently across $i = 1, \ldots, m$. The observed data are ${\mathbf{y}_i}$ and the missing data are ${\boldsymbol{\gamma}_i}$.

### Q2.1 Complete Log-Likelihood

The complete data density factors as
$$f(\mathbf{y}_i, \boldsymbol{\gamma}_i \mid \boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2) = f(\mathbf{y}_i \mid \boldsymbol{\gamma}_i, \boldsymbol{\beta}, \sigma^2) f(\boldsymbol{\gamma}_i \mid \boldsymbol{\Sigma}).$$
Taking logs and summing over $i$, the complete log-likelihood is
$$\ell_c(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2) = \sum_{i=1}^m \log f(\mathbf{y}_i \mid \boldsymbol{\gamma}_i, \boldsymbol{\beta}, \sigma^2) + \sum_{i=1}^m \log f(\boldsymbol{\gamma}_i \mid \boldsymbol{\Sigma}).$$
Substituting the Gaussian densities:
$$\ell_c = -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^m \|\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta} - \mathbf{Z}_i\boldsymbol{\gamma}_i\|^2$$ $$- \frac{m}{2}\log|2\pi\boldsymbol{\Sigma}| - \frac{1}{2}\sum_{i=1}^m \boldsymbol{\gamma}_i^T\boldsymbol{\Sigma}^{-1}\boldsymbol{\gamma}_i,$$
where $N = \sum_{i=1}^m n_i$ is the total number of observations.

### Q2.2 The Q Function (E-step)

The $Q$ function is the conditional expectation of the complete log-likelihood given the observed data, evaluated at the current parameter values $(\boldsymbol{\beta}^{(t)}, \boldsymbol{\Sigma}^{(t)}, \sigma^{2(t)})$:
$$Q(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2 \mid \boldsymbol{\beta}^{(t)}, \boldsymbol{\Sigma}^{(t)}, \sigma^{2(t)}) = E\left[\ell_c(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2) \mid \mathbf{y}, \boldsymbol{\beta}^{(t)}, \boldsymbol{\Sigma}^{(t)}, \sigma^{2(t)}\right].$$
Since the expectation only acts on terms involving the random effects, we need the following two posterior quantities for each $i$, computed under the current parameters:
$$\tilde{\boldsymbol{\gamma}}_i^{(t)} = E[\boldsymbol{\gamma}_i \mid \mathbf{y}_i, \boldsymbol{\beta}^{(t)}, \boldsymbol{\Sigma}^{(t)}, \sigma^{2(t)}] = \boldsymbol{\Sigma}^{(t)}\mathbf{Z}_i^T\left(\mathbf{Z}_i\boldsymbol{\Sigma}^{(t)}\mathbf{Z}_i^T + \sigma^{2(t)}\mathbf{I}\right)^{-1}(\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta}^{(t)}),$$
$$\widetilde{\boldsymbol{\gamma}_i\boldsymbol{\gamma}_i^T}^{(t)} = E[\boldsymbol{\gamma}_i\boldsymbol{\gamma}_i^T \mid \mathbf{y}_i, \boldsymbol{\beta}^{(t)}, \boldsymbol{\Sigma}^{(t)}, \sigma^{2(t)}] = \mathbf{V}_i^{(t)} + \tilde{\boldsymbol{\gamma}}_i^{(t)}\tilde{\boldsymbol{\gamma}}_i^{(t)T},$$
where the posterior covariance is
$$\mathbf{V}_i^{(t)} = \text{Var}(\boldsymbol{\gamma}_i \mid \mathbf{y}_i, \boldsymbol{\beta}^{(t)}, \boldsymbol{\Sigma}^{(t)}, \sigma^{2(t)}) = \boldsymbol{\Sigma}^{(t)} - \boldsymbol{\Sigma}^{(t)}\mathbf{Z}_i^T\left(\mathbf{Z}_i\boldsymbol{\Sigma}^{(t)}\mathbf{Z}_i^T + \sigma^{2(t)}\mathbf{I}\right)^{-1}\mathbf{Z}_i\boldsymbol{\Sigma}^{(t)}.$$
These follow directly from Q1. Expanding the quadratic in $\ell_c$ and taking expectations term by term:
$$E\left[\|\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta} - \mathbf{Z}_i\boldsymbol{\gamma}_i\|^2 \mid \mathbf{y}_i\right] = \|\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta} - \mathbf{Z}_i\tilde{\boldsymbol{\gamma}}_i^{(t)}\|^2 + \text{tr}\left(\mathbf{Z}_i\mathbf{V}_i^{(t)}\mathbf{Z}_i^T\right),$$
$$E\left[\boldsymbol{\gamma}_i^T\boldsymbol{\Sigma}^{-1}\boldsymbol{\gamma}_i \mid \mathbf{y}_i\right] = \text{tr}\left(\boldsymbol{\Sigma}^{-1}\widetilde{\boldsymbol{\gamma}_i\boldsymbol{\gamma}_i^T}^{(t)}\right).$$
Therefore the $Q$ function is
$$Q = -\frac{N}{2}\log\sigma^2 - \frac{1}{2\sigma^2}\sum_{i=1}^m\left[\|\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta} - \mathbf{Z}_i\tilde{\boldsymbol{\gamma}}_i^{(t)}\|^2 + \text{tr}\left(\mathbf{Z}_i\mathbf{V}_i^{(t)}\mathbf{Z}_i^T\right)\right]$$ $$- \frac{m}{2}\log|\boldsymbol{\Sigma}| - \frac{1}{2}\sum_{i=1}^m\text{tr}\left(\boldsymbol{\Sigma}^{-1}\widetilde{\boldsymbol{\gamma}_i\boldsymbol{\gamma}_i^T}^{(t)}\right) + \text{const}.$$

### Q2.3 M-step Updates

We maximize $Q$ over $(\boldsymbol{\beta}, \boldsymbol{\Sigma}, \sigma^2)$ separately, since the parameters separate across terms.

**Update for $\boldsymbol{\beta}$**

The only term in $Q$ involving $\boldsymbol{\beta}$ is
$$-\frac{1}{2\sigma^2}\sum_{i=1}^m\|\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta} - \mathbf{Z}_i\tilde{\boldsymbol{\gamma}}_i^{(t)}\|^2.$$
This is a weighted least squares problem in $\boldsymbol{\beta}$. Taking the derivative with respect to $\boldsymbol{\beta}$ and setting it to zero:
$$\sum_{i=1}^m \mathbf{X}_i^T\left(\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta} - \mathbf{Z}_i\tilde{\boldsymbol{\gamma}}_i^{(t)}\right) = \mathbf{0},$$
which gives
$$\boldsymbol{\beta}^{(t+1)} = \left(\sum_{i=1}^m \mathbf{X}_i^T\mathbf{X}i\right)^{-1}\sum_{i=1}^m \mathbf{X}_i^T\left(\mathbf{y}_i - \mathbf{Z}_i\tilde{\boldsymbol{\gamma}}_i^{(t)}\right).$$

**Update for $\boldsymbol{\Sigma}$**

The terms in $Q$ involving $\boldsymbol{\Sigma}$ are
$$-\frac{m}{2}\log|\boldsymbol{\Sigma}| - \frac{1}{2}\text{tr}\left(\boldsymbol{\Sigma}^{-1}\sum_{i=1}^m\widetilde{\boldsymbol{\gamma}_i\boldsymbol{\gamma}_i^T}^{(t)}\right).$$
This is the log-likelihood of a zero-mean Gaussian with covariance $\boldsymbol{\Sigma}$, so by the standard result:
$$\boldsymbol{\Sigma}^{(t+1)} = \frac{1}{m}\sum_{i=1}^m\widetilde{\boldsymbol{\gamma}_i\boldsymbol{\gamma}_i^T}^{(t)} = \frac{1}{m}\sum_{i=1}^m\left(\mathbf{V}_i^{(t)} + \tilde{\boldsymbol{\gamma}}_i^{(t)}\tilde{\boldsymbol{\gamma}}_i^{(t)T}\right).$$

**Update for $\sigma^2$**

The terms in $Q$ involving $\sigma^2$ are
$$-\frac{N}{2}\log\sigma^2 - \frac{1}{2\sigma^2}\sum_{i=1}^m\left[\|\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta}^{(t+1)} - \mathbf{Z}_i\tilde{\boldsymbol{\gamma}}_i^{(t)}\|^2 + \text{tr}\left(\mathbf{Z}_i\mathbf{V}_i^{(t)}\mathbf{Z}_i^T\right)\right].$$
Taking the derivative with respect to $\sigma^2$ and setting it to zero:
$$\sigma^{2(t+1)} = \frac{1}{N}\sum_{i=1}^m\left[\|\mathbf{y}_i - \mathbf{X}_i\boldsymbol{\beta}^{(t+1)} - \mathbf{Z}_i\tilde{\boldsymbol{\gamma}}_i^{(t)}\|^2 + \text{tr}\left(\mathbf{Z}_i\mathbf{V}_i^{(t)}\mathbf{Z}_i^T\right)\right].$$
Note that $\boldsymbol{\beta}^{(t+1)}$ is used here in place of $\boldsymbol{\beta}^{(t)}$, making this an ECM (Expectation Condi

## Q3. (20 pts) Objective of a single datum

We modify the code from HW6 to evaluate the objective, the conditional mean of $\boldsymbol{\gamma}$, and the conditional variance of $\boldsymbol{\gamma}$. Start-up code is provided below. You do _not_ have to use this code.

In [6]:
# define a type that holds an LMM datum
struct LmmObs{T <: AbstractFloat}
    # data
    y          :: Vector{T}
    X          :: Matrix{T}
    Z          :: Matrix{T}
    # posterior mean and variance of random effects γ
    μγ         :: Vector{T} # posterior mean of random effects
    νγ         :: Matrix{T} # posterior variance of random effects
    # pre-computed from data
    yty        :: T           # yᵀy
    xty        :: Vector{T}   # Xᵀy
    zty        :: Vector{T}   # Zᵀy
    xtx        :: Matrix{T}   # XᵀX
    ztx        :: Matrix{T}   # ZᵀX
    ztz        :: Matrix{T}   # ZᵀZ
    # updated each call (depend on β)
    rtr        :: Vector{T}   # ‖y - Xβ - Zγ̃‖² (scalar stored as 1-vector)
    xtr        :: Vector{T}   # Xᵀr
    ztr        :: Vector{T}   # Zᵀr
    # working arrays (scratch, never leave the function)
    ltztr      :: Vector{T}   # Lᵀ Zᵀr
    ltztzl     :: Matrix{T}   # Lᵀ ZᵀZ L  (= M - σ²I)
    storage_p  :: Vector{T}   # length-p scratch
    storage_q  :: Vector{T}   # length-q scratch; holds U⁻ᵀ Lᵀ Zᵀr after logl!
    storage_qq :: Matrix{T}   # q×q scratch; holds Cholesky U of M during logl!
    ltzt       :: Matrix{T}   # Lᵀ Zᵀ  (q×n); pre-allocate to avoid alloc in trmm path
end

"""
    LmmObs(y::Vector, X::Matrix, Z::Matrix)

Create an LMM datum of type `LmmObs`.
"""
function LmmObs(
    y::Vector{T},
    X::Matrix{T},
    Z::Matrix{T}) where T <: AbstractFloat
    n, p, q = size(X, 1), size(X, 2), size(Z, 2)
    μγ         = Vector{T}(undef, q)
    νγ         = Matrix{T}(undef, q, q)
    yty        = abs2(norm(y))
    xty        = transpose(X) * y
    zty        = transpose(Z) * y
    xtx        = transpose(X) * X
    ztx        = transpose(Z) * X
    ztz        = transpose(Z) * Z
    rtr        = Vector{T}(undef, 1)
    xtr        = Vector{T}(undef, p)
    ztr        = similar(zty)
    ltztr      = similar(zty)
    ltztzl     = similar(ztz)
    storage_p  = similar(xtr)
    storage_q  = Vector{T}(undef, q)
    storage_qq = similar(ztz)
    ltzt       = Matrix{T}(undef, q, n)   # the added pre-allocation
    LmmObs(y, X, Z, μγ, νγ,
        yty, xty, zty, xtx, ztx, ztz,
        rtr, xtr, ztr, ltztr, ltztzl,
        storage_p, storage_q, storage_qq, ltzt)
end

"""
    logl!(obs::LmmObs, β, Σ, L, σ², updater = false)

Evaluate the log-likelihood of a single LMM datum at parameter values `β`, `Σ`, 
and `σ²`. The lower triangular Cholesky factor `L` of `Σ` must be supplied too.
The fields `obs.μγ` and `obs.νγ` are overwritten by the posterior mean and 
posterior variance of random effects. If `updater==true`, fields `obs.ztr`, 
`obs.xtr`, and `obs.rtr` are updated according to input parameter values. 
Otherwise, it assumes these three fields are pre-computed. 
"""
function logl!(
        obs     :: LmmObs{T}, 
        β       :: Vector{T}, 
        Σ       :: Matrix{T},
        L       :: Matrix{T},
        σ²      :: T,
        updater :: Bool = false
        ) where T <: AbstractFloat
    n, p, q = size(obs.X, 1), size(obs.X, 2), size(obs.Z, 2)
    σ²inv   = inv(σ²)
    ####################
    # Evaluate objective
    ####################
    # form the q-by-q matrix: Lt Zt Z L
    copy!(obs.ltztzl, obs.ztz)
    BLAS.trmm!('L', 'L', 'T', 'N', T(1), L, obs.ltztzl) # O(q^3)
    BLAS.trmm!('R', 'L', 'N', 'N', T(1), L, obs.ltztzl) # O(q^3)        
    # form the q-by-q matrix: M = σ² I + Lt Zt Z L
    copy!(obs.storage_qq, obs.ltztzl)
    @inbounds for j in 1:q
        obs.storage_qq[j, j] += σ²
    end
    LAPACK.potrf!('U', obs.storage_qq) # O(q^3)
    # Zt * res
    updater && BLAS.gemv!('N', T(-1), obs.ztx, β, T(1), copy!(obs.ztr, obs.zty)) # O(pq)
    # Lt * (Zt * res)
    BLAS.trmv!('L', 'T', 'N', L, copy!(obs.ltztr, obs.ztr))    # O(q^2)
    # storage_q = (Mchol.U') \ (Lt * (Zt * res))
    BLAS.trsv!('U', 'T', 'N', obs.storage_qq, copy!(obs.storage_q, obs.ltztr)) # O(q^3)
    # Xt * res = Xt * y - Xt * X * β
    updater && BLAS.gemv!('N', T(-1), obs.xtx, β, T(1), copy!(obs.xtr, obs.xty))
    # l2 norm of residual vector
    updater && (obs.rtr[1] = obs.yty - dot(obs.xty, β) - dot(obs.xtr, β))
    # assemble pieces
    logl::T = n * log(2π) + (n - q) * log(σ²) # constant term
    @inbounds for j in 1:q # log det term
        logl += 2log(obs.storage_qq[j, j])
    end
    qf    = abs2(norm(obs.storage_q)) # quadratic form term
    logl += (obs.rtr[1] - qf) * σ²inv 
    logl /= -2
    ######################################
    # Evaluate posterior mean and variance
    ######################################    
    # With A = U⁻ᵀ Lᵀ  (q×q), we have:
    #   μγ = LM⁻¹Lᵀ Zᵀr = AᵀA Zᵀr = Aᵀ * storage_q
    #   Vγ = σ² LM⁻¹Lᵀ  = σ² AᵀA
    # Both are zero-allocation: A is formed in obs.νγ via trsm,
    # μγ via gemv, νγ via syrk. obs.storage_qq is safe to reuse here.
    #
    # Step 1: form A = U⁻ᵀ Lᵀ in obs.νγ
    @inbounds for j in 1:q, i in 1:q
        obs.νγ[i, j] = L[j, i]          # νγ ← Lᵀ
    end
    BLAS.trsm!('L', 'U', 'T', 'N', T(1), obs.storage_qq, obs.νγ)  # νγ ← U⁻ᵀ Lᵀ = A
    # Step 2: μγ = Aᵀ * storage_q  (storage_q = U⁻ᵀ Lᵀ Zᵀr, already computed)
    BLAS.gemv!('T', T(1), obs.νγ, obs.storage_q, T(0), obs.μγ)
    # Step 3: νγ = σ² AᵀA  (symmetric rank-q, upper triangle → full copy)
    BLAS.syrk!('U', 'T', T(σ²), obs.νγ, T(0), obs.storage_qq)
    @inbounds for j in 1:q
        for i in 1:j
            obs.νγ[i, j] = obs.storage_qq[i, j]
            obs.νγ[j, i] = obs.storage_qq[i, j]
        end
    end
    ###################
    # Return
    ###################        
    return logl
end

logl!

It is a good idea to test correctness and efficiency of the single datum objective/posterior mean/var evaluator here. It's the same test datum as in HW3 and HW6.

In [7]:
Random.seed!(257)

# dimension
n, p, q = 2000, 5, 3
# predictors
X = [ones(n) randn(n, p - 1)]
Z = [ones(n) randn(n, q - 1)]
# parameter values
β  = [2.0; -1.0; rand(p - 2)]
σ² = 1.5
Σ  = fill(0.1, q, q) + 0.9I # compound symmetry 
L  = Matrix(cholesky(Symmetric(Σ)).L)
# generate y
y  = X * β + Z * rand(MvNormal(Σ)) + sqrt(σ²) * randn(n)

# form the LmmObs object
obs = LmmObs(y, X, Z);

### Correctness

In [8]:
@show logl = logl!(obs, β, Σ, L, σ², true)
@show obs.μγ
@show obs.νγ;

logl = logl!(obs, β, Σ, L, σ², true) = -3261.3155007293944
obs.μγ = [0.10535698322857485, -0.2500949429916454, -1.4650548764788478]
obs.νγ = [0.000749435639578661 -1.2183420464422502e-6 -2.1767836829417426e-6; -1.2183420464422502e-6 0.0007542331466357767 2.1553464612468753e-5; -2.1767836829417426e-6 2.1553464612468753e-5 0.0007465271344917229]


You will lose all 20 points if following statement throws `AssertionError`.

In [9]:
@assert abs(logl - (-3261.315500729395)) < 1e-4
@assert norm(obs.μγ - [0.10535698322855516, 
        -0.2500949429916384, -1.4650548764788438]) < 1e-4
@assert norm(obs.νγ - [
        0.0007494356392878654 -1.2183420962445732e-6 -2.1767837161125168e-6; 
        -1.2183420773917486e-6 0.0007542331467495387 2.1553464604820387e-5; 
        -2.176783695623851e-6 2.1553464614324293e-5 0.0007465271343914666
        ]) < 1e-4

### Efficiency

Benchmark for efficiency.

In [10]:
bm_obj = @benchmark logl!($obs, $β, $Σ, $L, $σ², true)

BenchmarkTools.Trial: 10000 samples with 174 evaluations per sample.
 Range (min … max):  609.195 ns …  3.253 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     635.658 ns              ┊ GC (median):    0.00%
 Time  (mean ± σ):   642.724 ns ± 37.510 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

          ▅█▅▆▂                                                 
  ▁▁▁▁▂▃▅███████▆▅▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  609 ns          Histogram: frequency by time          749 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

My median runt time is 800ns. You will get full credit if the median run time is within 10μs. The points you will get are

In [11]:
clamp(10 / (median(bm_obj).time / 1e3) * 10, 0, 10)

10.0

In [ ]:
# check for type stability
@code_warntype logl!(obs, β, Σ, L, σ²)

MethodInstance for logl!(::LmmObs{Float64}, ::Vector{Float64}, ::Matrix{Float64}, ::Matrix{Float64}, ::Float64)
  from logl!(obs::LmmObs{T}, β::Vector{T}, Σ::Matrix{T}, L::Matrix{T}, σ²::T) where T<:AbstractFloat @ Main In[6]:73
Static Parameters
  T = Float64
Arguments
  #self#::Core.Const(Main.logl!)
  obs::LmmObs{Float64}
  β::Vector{Float64}
  Σ::Matrix{Float64}
  L::Matrix{Float64}
  σ²::Float64
Body::Float64
1 ─ %1 = (#self#)(obs, β, Σ, L, σ², false)::Float64
└──      return %1



In [13]:
using Profile

# Profile.clear()
@profile for i in 1:10000; logl!(obs, β, Σ, L, σ²); end
Profile.print(format=:flat)

 Count  Overhead File                    Line Function
 =====  ======== ====                    ==== ========
     6         5 In[13]                    4 macro expansion
     6         0 In[13]                    4 top-level scope
     1         0 In[6]                    81 logl!(obs::LmmObs{Float64}, β::Ve…
     1         0 In[6]                   107 logl!(obs::LmmObs{Float64}, β::Ve…
     6         0 @Base/]8;;file://yingxindeMacBook-Pro-254.local/Users/yingxin/.julia/juliaup/julia-1.12.5%2B0.aarch64.apple.darwin14/Julia-1.12.app/Contents/Resources/julia/share/julia/base/boot.jl\boot.jl]8;;\           489 eval(m::Module, e::Any)
     1         1 @Base/]8;;file://yingxindeMacBook-Pro-254.local/Users/yingxin/.julia/juliaup/julia-1.12.5%2B0.aarch64.apple.darwin14/Julia-1.12.app/Contents/Resources/julia/share/julia/base/float.jl\float.jl]8;;\          497 *
     6         0 @Base/]8;;file://yingxindeMacBook-Pro-254.local/Users/yingxin/.julia/juliaup/julia-1.12.5%2B0.aarch64.a

## Q4. LmmModel type

We modify the `LmmModel` type in HW6 to hold all data points, model parameters, and intermediate arrays.

In [14]:
# define a type that holds LMM model (data + parameters)
struct LmmModel{T <: AbstractFloat}
    # data
    data :: Vector{LmmObs{T}}
    # parameters
    β      :: Vector{T}
    Σ      :: Matrix{T}
    L      :: Matrix{T}
    σ²     :: Vector{T}    
    # pre-computed model-level quantities (computed once in constructor)
    xty    :: Vector{T}      # Σᵢ Xᵢᵀyᵢ
    xtxinv :: Matrix{T}      # (Σᵢ Xᵢᵀ Xᵢ)⁻¹  — used for β update and init_ls!
    # model-level accumulators (zeroed and re-filled each EM iteration)
    xtr    :: Vector{T}      # Σᵢ Xᵢᵀr̃ᵢ  where r̃ᵢ = yᵢ - Xᵢβ - Zᵢμ̃γᵢ
    # used by init_ls! for the LS estimate of Σ
    ztr2   :: Vector{T}      # vec(Σᵢ Zᵢᵀrᵢ ⊗ Zᵢᵀrᵢ),  length q²
    ztz2   :: Matrix{T}      # Σᵢ ZᵢᵀZᵢ ⊗ ZᵢᵀZᵢ,       size q²×q²
end

"""
    LmmModel(data::Vector{LmmObs})

Create an LMM model that contains data and parameters.
"""
function LmmModel(obsvec::Vector{LmmObs{T}}) where T <: AbstractFloat
    # dims
    p      = size(obsvec[1].X, 2)
    q      = size(obsvec[1].Z, 2)
    # parameters
    β      = Vector{T}(undef, p)
    Σ      = Matrix{T}(undef, q, q)
    L      = Matrix{T}(undef, q, q)
    σ²     = Vector{T}(undef, 1)    
    # pre-calculate Σᵢ Xᵢᵀyᵢ  and  Σᵢ Xᵢᵀ Xᵢ  (then invert)
    xty    = zeros(T, p)
    xtxinv = zeros(T, p, p)          # accumulate XᵀX here, then invert in-place
    @inbounds for obs in obsvec
        BLAS.axpy!(T(1), obs.xty, xty)
        BLAS.axpy!(T(1), obs.xtx, xtxinv)
    end
    # invert XᵀX via Cholesky (XᵀX is symmetric positive definite)
    LAPACK.potrf!('U', xtxinv)       # xtxinv ← U  s.t.  UᵀU = XᵀX
    LAPACK.potri!('U', xtxinv)       # xtxinv ← upper triangle of (XᵀX)⁻¹
    LinearAlgebra.copytri!(xtxinv, 'U')   # fill lower triangle
    # model-level accumulators (sized, values set each iteration)
    xtr    = similar(xty)
    ztr2   = Vector{T}(undef, q * q)      # q² = abs2(q)
    ztz2   = Matrix{T}(undef, q * q, q * q)
    LmmModel(obsvec, β, Σ, L, σ², xty, xtxinv, xtr, ztr2, ztz2)
end

LmmModel

## Q5. Implement EM update

Let's write the key function `update_em!` that performs one iteration of EM update.

In [15]:
"""
    update_em!(m::LmmModel, updater::Bool = false)

Perform one iteration of EM update. It returns the log-likelihood calculated 
from input `m.β`, `m.Σ`, `m.L`, and `m.σ²`. These fields are then overwritten 
by the next EM iterate. The fields `m.data[i].xtr`, `m.data[i].ztr`, and 
`m.data[i].rtr` are updated according to the resultant `m.β`. If `updater==true`, 
the function first updates `m.data[i].xtr`, `m.data[i].ztr`, and 
`m.data[i].rtr` according to `m.β`. If `updater==false`, it assumes these fields 
are pre-computed.
"""
function update_em!(m::LmmModel{T}, updater::Bool = false) where T <: AbstractFloat
    p, q  = size(m.data[1].X, 2), size(m.data[1].Z, 2)
    logl  = zero(T)
    ntotal = 0
 
    # Accumulators (model-level, zeroed once before the loop)
    fill!(m.xtr,  T(0))   # will hold Σᵢ Xᵢᵀ Zᵢ μ̃γᵢ  temporarily, then β rhs
    fill!(m.Σ,    T(0))   # will accumulate Σᵢ (Ṽγᵢ + μ̃γᵢ μ̃γᵢᵀ)
    σ²new = zero(T)        # will accumulate numerator of σ² update
 
    # E-step + partial M-step: loop over individuals
    @inbounds for obs in m.data
        ni = length(obs.y)
        ntotal += ni
 
        # evaluate logl at current (β, Σ, L, σ²) and compute
        #     posterior mean obs.μγ and variance obs.νγ (E-step)
        logl += logl!(obs, m.β, m.Σ, m.L, m.σ²[1], updater)
 
        # accumulate β M-step rhs: Σᵢ XᵢᵀZᵢ μ̃γᵢ
        #     ztx = ZᵀX (q×p), so XᵀZ = ztxᵀ (p×q)
        #     xtr += XᵢᵀZᵢ μ̃γᵢ  = ztxᵢᵀ * μ̃γᵢ
        BLAS.gemv!('T', T(1), obs.ztx, obs.μγ, T(1), m.xtr)  # O(pq)
 
        # accumulate σ² numerator:
        #     contribution = rtrᵢ - 2 μ̃γᵢᵀ(Zᵢᵀrᵢ) + μ̃γᵢᵀ(ZᵢᵀZᵢ)μ̃γᵢ + tr(Ṽγᵢ ZᵢᵀZᵢ)
        #     = rtrᵢ - 2 dot(μ̃γᵢ, ztrᵢ) + dot(μ̃γᵢ, ztz*μ̃γᵢ) + tr(νγᵢ * ztzᵢ)

        # storage_q ← ZᵢᵀZᵢ μ̃γᵢ
        BLAS.symv!('U', T(1), obs.ztz, obs.μγ, T(0), obs.storage_q)
        σ²new += obs.rtr[1]                        # rtrᵢ = ‖yᵢ - Xᵢβ‖²
        σ²new -= 2 * dot(obs.μγ, obs.ztr)          # -2 μ̃ᵀ Zᵀr
        σ²new += dot(obs.μγ, obs.storage_q)        # μ̃ᵀ ZᵀZ μ̃
        σ²new += dot(obs.νγ, obs.ztz)              # tr(ṼZᵀZ) = tr(ZᵀZṼ)
 
        # accumulate Σ M-step: Σ += Ṽγᵢ + μ̃γᵢ μ̃γᵢᵀ
        #     νγ is symmetric; axpy! treats it as a flat array (fine for accumulation)
        BLAS.axpy!(T(1), obs.νγ, m.Σ)             # Σ += Ṽγᵢ
        BLAS.ger!(T(1), obs.μγ, obs.μγ, m.Σ)      # Σ += μ̃γᵢ μ̃γᵢᵀ
    end

    # M-step: update β
    # β^(t+1) = xtxinv * (xty - Σᵢ XᵢᵀZᵢ μ̃γᵢ)
    #         = xtxinv * (xty - xtr)
    # Reuse m.xtr as the rhs vector: rhs = xty - xtr, store in m.xtr
    @inbounds for j in 1:p
        m.xtr[j] = m.xty[j] - m.xtr[j]    # m.xtr ← xty - Σ XᵀZμ̃
    end
    mul!(m.β, m.xtxinv, m.xtr)             # β^(t+1) = xtxinv * rhs
 
    # Update obs.xtr, obs.ztr, obs.rtr based on new β  (always done;
    # these are needed by the *next* call with updater=false)
    @inbounds for obs in m.data
        # obs.xtr = Xᵀy - XᵀX β  = Xᵀr
        BLAS.gemv!('N', T(-1), obs.xtx, m.β, T(1), copy!(obs.xtr, obs.xty))
        # obs.ztr = Zᵀy - ZᵀX β  = Zᵀr
        BLAS.gemv!('N', T(-1), obs.ztx, m.β, T(1), copy!(obs.ztr, obs.zty))
        # obs.rtr = yᵀy - 2βᵀXᵀy + βᵀXᵀXβ  = ‖r‖²
        obs.rtr[1] = obs.yty - dot(obs.xty, m.β) - dot(obs.xtr, m.β)
    end
 
    # M-step: update σ²
    # σ²^(t+1) = σ²new / N
    m.σ²[1] = σ²new / ntotal
 
    # M-step: update Σ and L
    # m.Σ currently holds Σᵢ (Ṽγᵢ + μ̃γᵢ μ̃γᵢᵀ); divide by m
    m_obs = length(m.data)
    m.Σ .*= inv(T(m_obs))                  # Σ^(t+1) = (1/m) Σᵢ (Ṽγᵢ + μ̃μ̃ᵀ)
    # recompute L = chol(Σ)
    copy!(m.L, m.Σ)
    LAPACK.potrf!('L', m.L)                # m.L ← lower Cholesky factor
    @inbounds for j in 2:q, i in 1:j-1    # zero strict upper triangle
        m.L[i, j] = zero(T)
    end
    # return log-likelihood at input parameter values
    logl
end

update_em!

## Q6. (30 pts) Test data

Let's generate a synthetic longitudinal data set (same as HW6) to test our algorithm.

In [16]:
Random.seed!(257)

# dimension
m      = 1000 # number of individuals
ns     = rand(1500:2000, m) # numbers of observations per individual
p      = 5 # number of fixed effects, including intercept
q      = 3 # number of random effects, including intercept
obsvec = Vector{LmmObs{Float64}}(undef, m)
# true parameter values
βtrue  = [0.1; 6.5; -3.5; 1.0; 5]
σ²true = 1.5
σtrue  = sqrt(σ²true)
Σtrue  = Matrix(Diagonal([2.0; 1.2; 1.0]))
Ltrue  = Matrix(cholesky(Symmetric(Σtrue)).L)
# generate data
for i in 1:m
    # first column intercept, remaining entries iid std normal
    X = Matrix{Float64}(undef, ns[i], p)
    X[:, 1] .= 1
    @views Distributions.rand!(Normal(), X[:, 2:p])
    # first column intercept, remaining entries iid std normal
    Z = Matrix{Float64}(undef, ns[i], q)
    Z[:, 1] .= 1
    @views Distributions.rand!(Normal(), Z[:, 2:q])
    # generate y
    y = X * βtrue .+ Z * (Ltrue * randn(q)) .+ σtrue * randn(ns[i])
    # form a LmmObs instance
    obsvec[i] = LmmObs(y, X, Z)
end
# form a LmmModel instance
lmm = LmmModel(obsvec);

### Correctness

Evaluate log-likelihood and gradient at the true parameter values.

In [17]:
copy!(lmm.β, βtrue)
copy!(lmm.Σ, Σtrue)
copy!(lmm.L, Ltrue)
lmm.σ²[1] = σ²true
@show obj1 = update_em!(lmm, true)
@show lmm.β
@show lmm.Σ
@show lmm.L
@show lmm.σ²
println()
@show obj2 = update_em!(lmm, false)
@show lmm.β
@show lmm.Σ
@show lmm.L
@show lmm.σ²

obj1 = update_em!(lmm, true) = -2.840600224113277e6
lmm.β = [0.09998887886409501, 6.500001802103847, -3.499087813220836, 1.0005635209262786, 5.000268027101281]
lmm.Σ = [1.8894514430318268 -0.0447736634828024 -0.012599156846234858; -0.0447736634828024 1.0989052393230383 0.05621230544677349; -0.012599156846234858 0.05621230544677349 1.0373370072880639]
lmm.L = [1.3745731857677956 0.0 0.0; -0.032572775277726056 1.0477806324005732 0.0; -0.009165868341304319 0.05336398283003458 1.0170571662803076]
lmm.σ² = [1.4997450576853244]

obj2 = update_em!(lmm, false) = -2.840594581606888e6
lmm.β = [0.09997873911624194, 6.500001872690205, -3.499086168891125, 1.0005643965940916, 5.0002685055523575]
lmm.Σ = [1.889354082567673 -0.04482291840116546 -0.012613626749128349; -0.04482291840116546 1.098763516543767 0.056302942465250916; -0.012613626749128349 0.056302942465250916 1.0374035360479197]
lmm.L = [1.3745377705132997 0.0 0.0; -0.03260944832707437 1.0477118594458932 0.0; -0.009176631606433039 0.05345333

1-element Vector{Float64}:
 1.4997434388368485

Test correctness. You will loss all 30 points if following code throws `AssertError`.

In [18]:
@assert abs(obj1 - (-2.8406002241132776e6)) < 1e-4
@assert abs(obj2 - (-2.840594581606412e6)) < 1e-4

### Efficiency

Test efficiency of EM update.

In [19]:
bm_emupdate = @benchmark update_em!($lmm, true) setup=(
    copy!(lmm.β, βtrue);
    copy!(lmm.Σ, Σtrue);
    copy!(lmm.L, Ltrue);
    lmm.σ²[1] = σ²true)

BenchmarkTools.Trial: 5609 samples with 1 evaluation per sample.
 Range (min … max):  854.250 μs …  1.522 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     887.250 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   890.904 μs ± 24.527 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

            ▂▄▅▆█▇█▇▄▆▅▆▄▂                                      
  ▁▁▁▁▂▄▅▆█████████████████▇▅▅▅▃▃▃▃▂▂▂▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▄
  854 μs          Histogram: frequency by time          966 μs <

 Memory estimate: 0 bytes, allocs estimate: 0.

My median run time is 1ms. You will get full credit if your median run time is within 10ms. The points you will get are

In [20]:
clamp(10 / (median(bm_emupdate).time / 1e6) * 10, 0, 10)

10.0

### Memory

You will lose 1 point for each 100 bytes memory allocation. So the points you will get is

In [21]:
clamp(10 - median(bm_emupdate).memory / 100, 0, 10)

10.0

## Q7. Starting point

We use the same least squares estimates as in HW6 as starting point. 

In [22]:
"""
    init_ls!(m::LmmModel)

Initialize parameters of a `LmmModel` object from the least squares estimate. 
`m.β`, `m.L`, and `m.σ²` are overwritten with the least squares estimates.
"""
function init_ls!(m::LmmModel{T}) where T <: AbstractFloat
    p, q = size(m.data[1].X, 2), size(m.data[1].Z, 2)
    # LS estimate for β
    mul!(m.β, m.xtxinv, m.xty)
    # LS etimate for σ2 and Σ
    rss, ntotal = zero(T), 0
    fill!(m.ztz2, 0)
    fill!(m.ztr2, 0)    
    @inbounds for i in eachindex(m.data)
        obs = m.data[i]
        ntotal += length(obs.y)
        # update Xt * res
        BLAS.gemv!('N', T(-1), obs.xtx, m.β, T(1), copy!(obs.xtr, obs.xty))
        # rss of i-th individual
        rss += obs.yty - dot(obs.xty, m.β) - dot(obs.xtr, m.β)
        # update Zi' * res
        BLAS.gemv!('N', T(-1), obs.ztx, m.β, T(1), copy!(obs.ztr, obs.zty))
        # Zi'Zi ⊗ Zi'Zi
        kron_axpy!(obs.ztz, obs.ztz, m.ztz2)
        # Zi'res ⊗ Zi'res
        kron_axpy!(obs.ztr, obs.ztr, m.ztr2)
    end
    m.σ²[1] = rss / ntotal
    # LS estimate for Σ = LLt
    LAPACK.potrf!('U', m.ztz2)
    BLAS.trsv!('U', 'T', 'N', m.ztz2, m.ztr2)
    BLAS.trsv!('U', 'N', 'N', m.ztz2, m.ztr2)
    copyto!(m.Σ, m.ztr2)
    copy!(m.L, m.Σ)
    LAPACK.potrf!('L', m.L)
    for j in 2:q, i in 1:j-1
        m.L[i, j] = 0
    end
    m
end

"""
    kron_axpy!(A, X, Y)

Overwrite `Y` with `A ⊗ X + Y`. Same as `Y += kron(A, X)` but
more memory efficient.
"""
function kron_axpy!(
        A::AbstractVecOrMat{T},
        X::AbstractVecOrMat{T},
        Y::AbstractVecOrMat{T}
        ) where T <: Real
    m, n = size(A, 1), size(A, 2)
    p, q = size(X, 1), size(X, 2)
    @assert size(Y, 1) == m * p
    @assert size(Y, 2) == n * q
    @inbounds for j in 1:n
        coffset = (j - 1) * q
        for i in 1:m
            a = A[i, j]
            roffset = (i - 1) * p            
            for l in 1:q
                r = roffset + 1
                c = coffset + l
                for k in 1:p                
                    Y[r, c] += a * X[k, l]
                    r += 1
                end
            end
        end
    end
    Y
end

kron_axpy!

In [23]:
init_ls!(lmm)
@show lmm.β
@show lmm.Σ
@show lmm.L
@show lmm.σ²

lmm.β = [0.07541303330100589, 6.500495707221947, -3.499669121836418, 0.9981512961737457, 5.001446434950227]
lmm.Σ = [1.8834379287447671 -0.05009003305573787 -0.00994616378562188; -0.05009003305573787 1.116177412921803 0.05590703591158548; -0.00994616378562188 0.055907035911585475 1.01982122703683]
lmm.L = [1.372384031073215 0.0 0.0; -0.03649855428335688 1.0558623340459814 0.0; -0.007247361933994454 0.052698648189682736 1.0084599918989388]
lmm.σ² = [5.521004671916751]


1-element Vector{Float64}:
 5.521004671916751

## Q8. Estimation by EM

We write a function `fit!` that implements the EM algorithm for estimating LMM.

In [24]:
"""
    fit!(m::LmmModel)

Fit an `LmmModel` object by MLE using a EM algorithm. Start point 
should be provided in `m.β`, `m.σ²`, `m.L`.
"""
function fit!(
        m       :: LmmModel;
        maxiter :: Integer       = 10_000,
        ftolrel :: AbstractFloat = 1e-12,
        prtfreq :: Integer       = 0
    )
    obj = update_em!(m, true)
    for iter in 0:maxiter
        obj_old = obj
        # EM update
        obj = update_em!(m, false)
        # print obj
        prtfreq > 0 && rem(iter, prtfreq) == 0 && println("iter=$iter, obj=$obj")
        # check monotonicity
        obj < obj_old && (@warn "monotonicity violated")
        # check convergence criterion
        (obj - obj_old) < ftolrel * (abs(obj_old) + 1) && break
        # warning about non-convergence
        iter == maxiter && (@warn "maximum iterations reached")
    end
    m
end


fit!

## Q9. (20 pts) Test drive

Now we can run our EM algorithm to compute the MLE.

In [25]:
# initialize from least squares
init_ls!(lmm)

@time fit!(lmm, prtfreq = 1);

println("objective value at solution: ", update_em!(lmm)); println()
println("solution values:")
@show lmm.β
@show lmm.σ²
@show lmm.L * transpose(lmm.L)

iter=0, obj=-2.840603672460753e6
iter=1, obj=-2.8405944513528817e6
iter=2, obj=-2.840594451324194e6
iter=3, obj=-2.8405944513229723e6
  0.043922 seconds (38.38 k allocations: 2.064 MiB, 86.56% compilation time)
objective value at solution: -2.8405944513217555e6

solution values:
lmm.β = [0.07542326891769854, 6.500001873356459, -3.4990861656168297, 1.0005643965158202, 5.00026850622348]
lmm.σ² = [1.4997434361363529]
lmm.L * transpose(lmm.L) = [1.8887848621468006 -0.0440393939936057 -0.01185105541105632; -0.0440393939936057 1.0987632916039871 0.056303089937154285; -0.01185105541105632 0.056303089937154285 1.0374036122246348]


3×3 Matrix{Float64}:
  1.88878    -0.0440394  -0.0118511
 -0.0440394   1.09876     0.0563031
 -0.0118511   0.0563031   1.0374

### Correctness

You get 10 points if the following code does not throw `AssertError`.

In [26]:
# objective at solution should be close enough to the optimal
@assert update_em!(lmm) > -2.84060e6

### Efficiency

My median run time 5ms. You get 10 points if your median run time is within 1s.

In [27]:
bm_em = @benchmark fit!($lmm) setup = (init_ls!(lmm))

BenchmarkTools.Trial: 1154 samples with 1 evaluation per sample.
 Range (min … max):  3.909 ms …  4.723 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     4.069 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.087 ms ± 99.092 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

         ▁▁▄▂▅▆▄▂▇▇▇▅█▂▅▃  ▁                                  
  ▂▃▃▄▅▅▆█████████████████▆█▆▅▇▅▆▄▄▄▅▃▃▆▃▃▃▃▂▃▃▃▃▂▁▂▂▂▁▁▁▁▂▂ ▄
  3.91 ms        Histogram: frequency by time        4.44 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [28]:
# this is the points you get
clamp(1 / (median(bm_em).time / 1e9) * 10, 0, 10)

10.0

## Q10. (10 pts) EM vs Newton type algorithms

Contrast EM algorithm to the Newton type algorithms (gradient free, gradient based, using Hessian) in HW6, in terms of the stability, convergence rate (how fast the algorithm is converging),  final objective value, total run time, derivation, and implementation efforts. 

**Answer:**

**Stability**. EM is usually very stable because it is monotone by construction: each iteration does not decrease the observed log-likelihood. Newton-type methods can converge faster, but they may require line search or step-size safeguards to avoid divergence.

**Convergence rate**. EM usually has linear convergence, and it can be slow when the missing information is large. Newton’s method has quadratic convergence near the optimum when the Hessian is well behaved. Gradient-based methods are usually between these two in speed.

**Final objective value**. In this example, EM and the Newton-type algorithms converge to essentially the same maximum log-likelihood, about -2,840,594.45, because they optimize the same marginal likelihood.

**Total run time**. In this example, EM is faster overall, about 4 ms, compared with about 0.05 s for the HW6 Newton-type implementation. However, this is problem-dependent. Newton-type methods can be faster when EM requires many iterations.

**Derivation effort**. EM requires deriving the conditional distribution of the random effects and the closed-form M-step updates. Newton-type methods require derivatives of the marginal log-likelihood, including log-determinant and matrix inverse terms, which is more algebraically complicated.

**Implementation effort**. EM is relatively easy to implement once the E-step quantities are available, because the M-step has closed-form updates. Newton-type methods require a general optimizer and either hand-coded gradients/Hessians or automatic differentiation, so they are usually harder to debug.